# 🇻🇳 VIET CONTRACT AUDITOR — Phase 2: Knowledge Graph Indexing (v2)
**LLM Engine** : `Qwen2.5-32B-Instruct-AWQ` via **local vLLM server** (100% offline)  
**Embedding**  : `paraphrase-multilingual-MiniLM-L12-v2` (local, A100)  
**Hardware**   : NVIDIA A100 80 GB | 160 GB RAM  
**Input**      : `processed_legal_chunks.json` (256 chunks)  
**Output**     : LightRAG Knowledge Graph (.graphml) + VectorDB → Google Drive  

---
### ⚡ Luồng chạy lần đầu
```
Cell 1 → Cell 2 (cài deps + TỰ ĐỘNG restart) → chạy lại từ Cell 1 → Cell 3 → ...
```
### ⚡ Luồng chạy lần sau (đã cài rồi)
```
Cell 1 → Cell 2 (tự skip) → Cell 3 → ... → Cell 14
```


## CELL 1 — LLM Model Configuration

In [7]:
# ============================================================
# CELL 1: Local vLLM Server Configuration
# ============================================================
import os
import torch

VLLM_PORT       = 8000
VLLM_BASE_URL   = f"http://localhost:{VLLM_PORT}/v1"
VLLM_API_KEY    = "EMPTY"
VLLM_MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct-AWQ"

API_KEY      = VLLM_API_KEY
MODEL_NAME   = VLLM_MODEL_NAME
API_BASE_URL = VLLM_BASE_URL

assert torch.cuda.is_available(), "GPU not available — switch to A100 runtime!"
gpu = torch.cuda.get_device_properties(0)
print(f"Cell 1 OK")
print(f"   GPU  : {gpu.name}")
print(f"   VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"   CUDA : {torch.version.cuda}")
print(f"   Model: {VLLM_MODEL_NAME}")


Cell 1 OK
   GPU  : NVIDIA A100-SXM4-80GB
   VRAM : 85.1 GB
   CUDA : 12.4
   Model: Qwen/Qwen2.5-32B-Instruct-AWQ


## CELL 2 — Install Dependencies (auto-restart nếu cần)
> **Lần đầu chạy:** Cell này sẽ cài thư viện rồi **tự động restart runtime**.  
> Sau khi restart, chạy lại từ Cell 1 — Cell 2 sẽ tự detect và **bỏ qua**.


In [ ]:
# ============================================================
# CELL 2: Smart Dependency Installer
#
# Vấn đề gốc rễ (Colab):
#   - numpy 1.26.x được pre-install, nhưng scipy/sklearn mới build cho numpy 2.x
#     → ValueError: numpy.dtype size changed (binary incompatibility)
#   - vllm 0.6.6 cần transformers>=4.45.2, tokenizers>=0.19.1
#     → AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
#   - Sau khi pip upgrade, Python vẫn giữ module cũ trong RAM
#     → Phải restart runtime để nạp lại binary mới
#
# Giải pháp: Kiểm tra version trước, chỉ cài nếu cần, dùng os._exit(0) để restart.
# ============================================================
import importlib, sys

def _version_tuple(v):
    return tuple(int(x) for x in v.split(".")[:3] if x.isdigit())

def _check_deps():
    """Trả về (ok: bool, reason: str)"""
    checks = [
        ("numpy",          "2.0.0",  lambda m: m.__version__),
        ("transformers",   "4.45.2", lambda m: m.__version__ if tuple(int(x) for x in m.__version__.split(".")[:2]) < (5,0) else "0.0.0"),
        ("tokenizers",     "0.19.1", lambda m: m.__version__),
        ("vllm",           "0.6.6",  lambda m: m.__version__),
        ("lightrag",       "0.0.1",  lambda m: getattr(m, "__version__", "0.0.1")),
        ("sentence_transformers", "2.0.0", lambda m: m.__version__),
        ("openai",         "1.0.0",  lambda m: m.__version__),
    ]
    for pkg, min_ver, get_ver in checks:
        try:
            mod = importlib.import_module(pkg)
            actual = get_ver(mod)
            if _version_tuple(actual) < _version_tuple(min_ver):
                return False, f"{pkg} {actual} < {min_ver}"
        except ImportError:
            return False, f"{pkg} not installed"
    return True, "all OK"

ok, reason = _check_deps()

if ok:
    print(f"All dependencies already installed correctly — skipping Cell 2.")
    print(f" Tiếp tục chạy Cell 3.")
else:
    print(f" Cần cài/nâng cấp: {reason}")
    print(f"Bắt đầu cài đặt theo thứ tự tối ưu...\n")

    # ── BƯỚC 1: numpy 2.x PHẢI được cài ĐẦU TIÊN ─────────────────────────────
    # Lý do: scipy, sklearn, cupy đều build binary với numpy 2.x ABI.
    # Nếu cài sau, binary mismatch vẫn xảy ra trong session hiện tại.
    print("[1/6] numpy >= 2.0 (nền tảng, phải upgrade trước)...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "numpy>=2.0", "--upgrade", "-q"], check=True)

    # ── BƯỚC 2: transformers + tokenizers (dependency của vLLM & AutoAWQ) ─────
    print("[2/6] transformers>=4.45.2, tokenizers>=0.19.1...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "transformers>=4.45.2,<5.0.0", "tokenizers>=0.19.1",
                    "--upgrade", "-q"], check=True)

    # ── BƯỚC 3: vLLM (pinned version cho CUDA 12.x Colab) ────────────────────
    print("[3/6] vllm==0.6.6.post1...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "vllm==0.6.6.post1", "-q"], check=True)

    # ── BƯỚC 4: AutoAWQ (cần transformers>=4.45 đã có ở bước 2) ─────────────
    print("[4/6] autoawq...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "autoawq", "-q"], check=True)

    # ── BƯỚC 5: LightRAG + sentence-transformers ──────────────────────────────
    print("[5/6] lightrag-hku, sentence-transformers...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "lightrag-hku", "sentence-transformers",
                    "-q"], check=True)

    # ── BƯỚC 6: Utilities ─────────────────────────────────────────────────────
    print("[6/6] openai, tenacity, aiohttp, networkx, tiktoken...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "openai", "tenacity", "aiohttp",
                    "networkx", "tiktoken", "-q"], check=True)

    print("\n" + "="*60)
    print("Cài đặt hoàn tất!")
    print("Đang tự động restart runtime để nạp binary mới...")
    print("   → Sau khi restart: chạy lại từ Cell 1.")
    print("   → Cell 2 sẽ tự detect và bỏ qua.")
    print("="*60)

    import os, time
    time.sleep(2)   # Cho output kịp flush ra màn hình
    os._exit(0)     # Hard restart — nạp lại toàn bộ binary


 Cần cài/nâng cấp: transformers 0.0.0 < 4.45.2
Bắt đầu cài đặt theo thứ tự tối ưu...

[1/6] numpy >= 2.0 (nền tảng, phải upgrade trước)...
[2/6] transformers>=4.45.2, tokenizers>=0.19.1...
[3/6] vllm==0.6.6.post1...
[4/6] autoawq...
[5/6] lightrag-hku, sentence-transformers...
[6/6] openai, tenacity, aiohttp, networkx, tiktoken...

Cài đặt hoàn tất!
Đang tự động restart runtime để nạp binary mới...
   → Sau khi restart: chạy lại từ Cell 1.
   → Cell 2 sẽ tự detect và bỏ qua.


## CELL 3 — GPU Memory Tracker

In [8]:
# ============================================================
# CELL 3: GPU Memory Tracker (thay thế DailyTokenBudget)
# ============================================================
import torch

class GPUMemoryTracker:
    @staticmethod
    def status() -> str:
        if not torch.cuda.is_available():
            return "No GPU"
        reserved = torch.cuda.memory_reserved(0) / 1e9
        total    = torch.cuda.get_device_properties(0).total_memory / 1e9
        pct      = 100 * reserved / total
        bar      = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        return f"[{bar}] {pct:.1f}% | {reserved:.1f}/{total:.1f} GB VRAM"

    @staticmethod
    def log(label: str = ""):
        tag = f" [{label}]" if label else ""
        print(f"  VRAM{tag}: {GPUMemoryTracker.status()}")

print("GPUMemoryTracker ready")


GPUMemoryTracker ready


## CELL 4 — Mount Google Drive & Initialize Storage

In [9]:
# ============================================================
# CELL 4: Mount Google Drive & Initialize Storage
# ============================================================
from google.colab import drive
from pathlib import Path

print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=True)

GDRIVE_BASE    = Path("/content/drive/MyDrive/viet_auditor")
WORKING_DIR    = GDRIVE_BASE / "lightrag_index"   # ← trỏ thẳng vào Drive
CHECKPOINT_DIR = GDRIVE_BASE / "checkpoints"       # ← nhất quán luôn
LOG_DIR        = Path("./viet_auditor/logs")       # ← log để local cũng được

for d in [GDRIVE_BASE, WORKING_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  📁 {d}")

gpu_tracker = GPUMemoryTracker()
print(f"\n Storage initialized | {gpu_tracker.status()}")

Mounting Google Drive...
Mounted at /content/drive
  📁 /content/drive/MyDrive/viet_auditor
  📁 /content/drive/MyDrive/viet_auditor/lightrag_index
  📁 /content/drive/MyDrive/viet_auditor/checkpoints
  📁 viet_auditor/logs

 Storage initialized | [░░░░░░░░░░░░░░░░░░░░] 0.6% | 0.5/85.1 GB VRAM


## CELL 5 — Load & Validate Input Data

In [10]:
# ============================================================
# CELL 5: Load processed_legal_chunks.json
# Upload file vào /content/ HOẶC để trong Google Drive.
# ============================================================
import json
from collections import defaultdict
from pathlib import Path

CANDIDATE_PATHS = [
    "/content/processed_legal_chunks.json",
    str(GDRIVE_BASE / "processed_legal_chunks.json"),
    "/content/drive/MyDrive/processed_legal_chunks.json",
]
CHUNKS_FILE = next((p for p in CANDIDATE_PATHS if Path(p).exists()), None)
assert CHUNKS_FILE, (
    " Không tìm thấy processed_legal_chunks.json\n"
    "   → Upload lên /content/ hoặc đặt trong Google Drive."
)
print(f" Found: {CHUNKS_FILE}")

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

if isinstance(raw_data, list):
    all_chunks = raw_data
elif isinstance(raw_data, dict) and "chunks" in raw_data:
    all_chunks = raw_data["chunks"]
else:
    all_chunks = []
    for key, val in raw_data.items():
        if isinstance(val, list):
            for chunk in val:
                if isinstance(chunk, dict):
                    chunk.setdefault("law_id", key)
                    all_chunks.append(chunk)

print(f" Loaded {len(all_chunks)} chunks")
by_law = defaultdict(list)
for c in all_chunks:
    by_law[c.get("law_id", "unknown")].append(c)
print("\nDistribution:")
for law_id, chunks in by_law.items():
    print(f"   {law_id}: {len(chunks)} chunks")

def get_chunk_text(chunk: dict) -> str:
    text    = chunk.get("content") or chunk.get("text") or chunk.get("chunk_text") or ""
    law_id  = chunk.get("law_id", "") or chunk.get("metadata", {}).get("law_id", "")
    article = chunk.get("article", chunk.get("dieu", "")) or chunk.get("metadata", {}).get("article_number", "")
    clause  = chunk.get("clause",  chunk.get("khoan", ""))
    header  = f"[Nguồn: {law_id}"
    if article: header += f" | Điều {article}"
    if clause:  header += f" | Khoản {clause}"
    return header + "]\n" + text

corpus = [get_chunk_text(c) for c in all_chunks]
print(f"\n Corpus ready: {len(corpus)} chunks | {gpu_tracker.status()}")


 Found: /content/drive/MyDrive/processed_legal_chunks.json
 Loaded 256 chunks

Distribution:
   unknown: 256 chunks

 Corpus ready: 256 chunks | [░░░░░░░░░░░░░░░░░░░░] 0.6% | 0.5/85.1 GB VRAM


## CELL 6 — Load Embedding Model
> ⚠️ Load embedding **TRƯỚC** khi khởi động vLLM để CUDA context được thiết lập trước (~0.5 GB VRAM).


In [11]:
# ============================================================
# CELL 6: Load Multilingual Embedding Model (local, zero cost)
# Phải chạy TRƯỚC Cell 6.5 (vLLM server).
# ============================================================
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM = 384

assert torch.cuda.is_available(), "GPU not available!"
print(f"Loading embedding model: {EMBED_MODEL_NAME}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
embed_model.eval()

test_emb = embed_model.encode(["Hợp đồng kinh tế Việt Nam."], normalize_embeddings=True)
print(f"\n Embedding model loaded!")
print(f"   Dim    : {test_emb.shape[1]}")
print(f"   Device : {next(embed_model.parameters()).device}")
gpu_tracker.log("after embedding model")


Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

 Embedding model loaded!
   Dim    : 384
   Device : cuda:0
  VRAM [after embedding model]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM


## CELL 6.5 — Start vLLM Local Server
> **Lần đầu:** Download model ~18 GB từ HuggingFace, có thể mất 10–20 phút.  
> **Lần sau:** Load từ cache, ~2–3 phút.


In [25]:
# ============================================================
# CELL 6.5: Start vLLM OpenAI-Compatible Local Server
#
# VRAM allocation (A100 80 GB):
#   Embedding model : ~0.5 GB
#   Qwen2.5-32B-AWQ : ~18–20 GB
#   KV cache (85%)  : ~50 GB
#   ─────────────────────────
#   Total           : ~70 GB / 80 GB
# ============================================================
import subprocess, requests, time, os
from pathlib import Path

def kill_existing_vllm():
    result = subprocess.run(
        ["pkill", "-f", "vllm.entrypoints.openai.api_server"],
        capture_output=True
    )
    if result.returncode == 0:
        print(" Killed previous vLLM process — waiting 5s...")
        time.sleep(5)

kill_existing_vllm()

LOG_FILE = LOG_DIR / "vllm_server.log"

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model",                    VLLM_MODEL_NAME,
    "--port",                     str(VLLM_PORT),
    "--host",                     "0.0.0.0",
    "--api-key",                  VLLM_API_KEY,
    "--max-model-len",            "32768",
    "--tensor-parallel-size",     "1",
    "--gpu-memory-utilization",   "0.85",
    "--dtype",                    "auto",
    "--trust-remote-code",
    "--disable-log-requests",
]

print(f" Starting vLLM server...")
print(f"   Model : {VLLM_MODEL_NAME}")
print(f"   Port  : {VLLM_PORT}")
print(f"   Log   : {LOG_FILE}")
gpu_tracker.log("before vLLM start")

with open(LOG_FILE, "w") as log_f:
    vllm_proc = subprocess.Popen(
        vllm_cmd,
        stdout=log_f,
        stderr=subprocess.STDOUT,
        preexec_fn=os.setsid
    )

HEALTH_URL    = f"http://localhost:{VLLM_PORT}/health"
MAX_WAIT_SEC  = 900
POLL_INTERVAL = 10
_download_logged = _weights_logged = False

print(f"\n⏳ Waiting for server (max {MAX_WAIT_SEC//60} min)...")
start_time = time.time()

for _ in range(MAX_WAIT_SEC // POLL_INTERVAL):
    elapsed = int(time.time() - start_time)

    if vllm_proc.poll() is not None:
        tail = open(LOG_FILE).readlines()[-30:]
        print("\n vLLM crashed! Last log:")
        for line in tail: print(f"   {line.rstrip()}")
        print("\n Nếu lỗi OOM, thử fallback model:")
        print("   VLLM_MODEL_NAME = 'Qwen/Qwen2.5-14B-Instruct'")
        raise RuntimeError("vLLM failed to start.")

    if LOG_FILE.exists():
        content = open(LOG_FILE).read()
        if "Downloading" in content and not _download_logged:
            print(f"   [{elapsed:4d}s]  Downloading model (~18 GB)...")
            _download_logged = True
        if "Loading model weights" in content and not _weights_logged:
            print(f"   [{elapsed:4d}s]   Loading weights into VRAM...")
            _weights_logged = True

    try:
        if requests.get(HEALTH_URL, timeout=3).status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass

    print(f"   [{elapsed:4d}s] Starting...", end="\r")
    time.sleep(POLL_INTERVAL)
else:
    raise TimeoutError(f"Server timeout after {MAX_WAIT_SEC}s. Log: {LOG_FILE}")

load_time = int(time.time() - start_time)
print(f"\n vLLM server ready! ({load_time}s)")
print(f"   Endpoint : {VLLM_BASE_URL}")

import openai as _oai
_models = [m.id for m in _oai.OpenAI(api_key=VLLM_API_KEY, base_url=VLLM_BASE_URL).models.list().data]
print(f"   Models   : {_models}")
assert any(VLLM_MODEL_NAME in m or m in VLLM_MODEL_NAME for m in _models), \
    f" Model not found: {_models}"

gpu_tracker.log("after vLLM load")
print("\n Model verified — ready for indexing!")


 Killed previous vLLM process — waiting 5s...
 Starting vLLM server...
   Model : Qwen/Qwen2.5-32B-Instruct-AWQ
   Port  : 8000
   Log   : viet_auditor/logs/vllm_server.log
  VRAM [before vLLM start]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM

⏳ Waiting for server (max 15 min)...
   [  80s]   Loading weights into VRAM...

 vLLM server ready! (130s)
   Endpoint : http://localhost:8000/v1
   Models   : ['Qwen/Qwen2.5-32B-Instruct-AWQ']
  VRAM [after vLLM load]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM

 Model verified — ready for indexing!


## CELL 7 — LightRAG LLM & Embedding Functions

In [13]:
# ============================================================
# CELL 7: LightRAG Wrapper Functions
#   llm_model_func  → local vLLM (no rate limits, no budget)
#   embedding_func  → local MiniLM (unchanged)
# ============================================================
import asyncio, openai, numpy as np, logging
from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type
)

logger = logging.getLogger(__name__)

aclient = openai.AsyncOpenAI(
    api_key=VLLM_API_KEY,
    base_url=VLLM_BASE_URL,
    timeout=180.0,
    max_retries=0,
)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=5, max=120),
    retry=retry_if_exception_type((
        openai.APIConnectionError,
        openai.APITimeoutError,
        openai.InternalServerError,
    )),
    reraise=True
)
async def llm_model_func(
    prompt: str,
    system_prompt: str | None = None,
    history_messages: list = [],
    **kwargs
) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    response = await aclient.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=kwargs.get("max_tokens", 2048),
        temperature=kwargs.get("temperature", 0.1),
        top_p=0.9,
    )
    return response.choices[0].message.content


async def embedding_func(texts: list[str]) -> np.ndarray:
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(
        None,
        lambda: embed_model.encode(texts, normalize_embeddings=True,
                                   batch_size=64, show_progress_bar=False)
    )


async def _smoke_test():
    print(" LLM smoke test...")
    resp = await llm_model_func(
        "Điều 1 Bộ luật Dân sự 2015 nói về điều gì? Trả lời 1 câu ngắn.",
        max_tokens=80
    )
    print(f"    Response: {resp.strip()}")
    gpu_tracker.log("after LLM call")

    emb = await embedding_func(["hợp đồng mua bán tài sản"])
    print(f"\n Embedding: shape={emb.shape} ")
    print("\n Both functions ready!")

await _smoke_test()


 LLM smoke test...
    Response: Điều 1 Bộ luật Dân sự 2015 quy định về phạm vi điều chỉnh và chủ thể của bộ luật này.
  VRAM [after LLM call]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM

 Embedding: shape=(1, 384) 

 Both functions ready!


## CELL 8 — Vietnamese Legal NER Prompts (LightRAG)

In [14]:
# ============================================================
# CELL 8: Override LightRAG Prompts — Vietnamese Legal NER
#
#  CRITICAL: Dùng định dạng tuple pipe-delimited của LightRAG.
# KHÔNG dùng JSON — parser sẽ silent fail, 0 entities extracted.
# ============================================================
from lightrag.prompt import PROMPTS

PROMPTS["DEFAULT_ENTITY_TYPES"] = [
    "CHỦ_THỂ", "HÀNH_VI", "QUYỀN_NGHĨA_VỤ", "CHẾ_TÀI", "VĂN_BẢN_PHÁP_LUẬT"
]

PROMPTS["entity_extraction"] = """-Goal-
Bạn là chuyên gia phân tích pháp lý Việt Nam. Nhiệm vụ: trích xuất tất cả thực thể pháp lý và mối quan hệ từ văn bản, sau đó tổng hợp thành danh sách.

-Steps-
1. Xác định tất cả thực thể. Với mỗi thực thể, trích xuất:
   - entity_name: Tên đầy đủ (viết hoa)
   - entity_type: Một trong các loại: {entity_types}
   - entity_description: Mô tả đầy đủ bao gồm trích dẫn điều khoản, ví dụ: "Bên mua có quyền yêu cầu giao hàng đúng hạn [Nguồn: 91/2015/qh13 | Điều 430]"
   Format: ('entity'{{tuple_delimiter}}<entity_name>{{tuple_delimiter}}<entity_type>{{tuple_delimiter}}<entity_description>)

2. Xác định tất cả quan hệ GIỮA các thực thể đã tìm được. Với mỗi quan hệ:
   - source_entity, target_entity: tên thực thể (phải khớp bước 1)
   - relationship_description: Mô tả quan hệ, kèm trích dẫn điều khoản
   - relationship_strength: Số từ 1-10
   Format: ('relationship'{{tuple_delimiter}}<source_entity>{{tuple_delimiter}}<target_entity>{{tuple_delimiter}}<relationship_description>{{tuple_delimiter}}<relationship_strength>)

3. Trả về danh sách tuple, dùng **{{record_delimiter}}** phân cách.

4. Kết thúc bằng {{completion_delimiter}}

-Ví dụ-
Văn bản: "[Nguồn: 91/2015/qh13 | Điều 430] Bên bán có nghĩa vụ giao tài sản đúng thời hạn đã thỏa thuận."

Output:
('entity'{{tuple_delimiter}}BÊN BÁN{{tuple_delimiter}}CHỦ_THỂ{{tuple_delimiter}}Bên bán trong hợp đồng mua bán [Nguồn: 91/2015/qh13 | Điều 430])
{{record_delimiter}}
('entity'{{tuple_delimiter}}GIAO TÀI SẢN ĐÚNG HẠN{{tuple_delimiter}}HÀNH_VI{{tuple_delimiter}}Hành vi giao tài sản đúng thời hạn thỏa thuận [Nguồn: 91/2015/qh13 | Điều 430])
{{record_delimiter}}
('relationship'{{tuple_delimiter}}BÊN BÁN{{tuple_delimiter}}GIAO TÀI SẢN ĐÚNG HẠN{{tuple_delimiter}}Bên bán có nghĩa vụ giao tài sản đúng hạn theo Điều 430 BLDS 2015{{tuple_delimiter}}9)
{{completion_delimiter}}

-Văn bản thực tế-
######################
{input_text}
######################
Output:
"""

PROMPTS["keywords_extraction"] = """-Mục tiêu-
Phân tích câu truy vấn pháp lý và trích xuất từ khóa theo hai cấp độ.

-Định dạng đầu ra- JSON thuần (không có backtick):
{{"high_level_keywords": ["chủ đề pháp lý cấp cao"], "low_level_keywords": ["điều khoản, thuật ngữ cụ thể"]}}

-Câu truy vấn-
{query}
"""

print(" Vietnamese Legal NER prompts applied!")
print(f"   Entity types : {PROMPTS['DEFAULT_ENTITY_TYPES']}")
print("   Format       : native LightRAG pipe-delimiter tuple")


 Vietnamese Legal NER prompts applied!
   Entity types : ['CHỦ_THỂ', 'HÀNH_VI', 'QUYỀN_NGHĨA_VỤ', 'CHẾ_TÀI', 'VĂN_BẢN_PHÁP_LUẬT']
   Format       : native LightRAG pipe-delimiter tuple


## CELL 9 — Initialize LightRAG

In [23]:
# ============================================================
# Initialize LightRAG Instance
# llm_model_max_async=4 (local vLLM, không có rate limit)
# ============================================================
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc

print("🔧 Initializing LightRAG...")

rag = LightRAG(
    working_dir=str(WORKING_DIR),
    llm_model_func=llm_model_func,
    llm_model_max_async=4,
    llm_model_kwargs={"temperature": 0.1, "max_tokens": 2048},
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=512,
        func=embedding_func
    ),
    embedding_batch_num=32,
    embedding_func_max_async=8,
    chunk_token_size=2400,
    chunk_overlap_token_size=100,
    tiktoken_model_name="gpt-4",
    enable_llm_cache=True,
)

# Gọi initialize_storages trước khi query
await rag.initialize_storages()
print(f"✅ LightRAG initialized + storages loaded")
gpu_tracker.log("after LightRAG init")


INFO: [] Loaded graph from /content/drive/MyDrive/viet_auditor/lightrag_index/graph_chunk_entity_relation.graphml with 2138 nodes, 2754 edges


🔧 Initializing LightRAG...
✅ LightRAG initialized + storages loaded
  VRAM [after LightRAG init]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM


## CELL 10 — Batched Indexing

In [ ]:
# ============================================================
# CELL 10: Batched Indexing với checkpoint + deduplication
# Inter-batch sleep: 2s (không có API rate limit)
# ============================================================
import torch, gc, json, hashlib, asyncio
from datetime import datetime

BATCH_SIZE      = 16
CHECKPOINT_FILE = CHECKPOINT_DIR / "indexing_progress.json"

def doc_hash(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()

def load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
        completed = set(data.get("completed_hashes", []))
        print(f"📋 Resumed: {len(completed)}/{len(corpus)} completed")
        return completed
    return set()

def save_checkpoint(completed_hashes: set, failed_texts: list = None):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({
            "completed_hashes": list(completed_hashes),
            "failed_texts":     failed_texts or [],
            "total":            len(corpus),
            "last_updated":     datetime.now().isoformat()
        }, f, indent=2)

async def run_indexing():
    print("⚙️  Syncing LightRAG storage...")
    await rag.initialize_storages()

    completed_hashes = load_checkpoint()
    remaining        = [t for t in corpus if doc_hash(t) not in completed_hashes]

    if not remaining:
        print("🎉 All chunks already indexed!")
        return

    batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
    print(f"🚀 {len(remaining)} chunks remaining | {len(batches)} batches")

    failed_texts = []
    for batch_num, batch in enumerate(batches, 1):
        print(f"\n📦 Batch {batch_num}/{len(batches)} (size: {len(batch)})")
        try:
            await rag.ainsert(batch)
            for t in batch:
                completed_hashes.add(doc_hash(t))
            save_checkpoint(completed_hashes, failed_texts)
            print(f"   ✅ Done. Total: {len(completed_hashes)}/{len(corpus)}")
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            failed_texts.extend(batch)
            save_checkpoint(completed_hashes, failed_texts)
            await asyncio.sleep(5)
            continue

        await asyncio.sleep(2)
        gpu_tracker.log(f"batch {batch_num}/{len(batches)}")
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n🏁 Indexing finished!")
    if failed_texts:
        print(f"   ⚠️  {len(failed_texts)} failed — run Cell 11 to retry.")
    else:
        print(f"   ✅ All chunks indexed successfully!")

await run_indexing()


INFO: [] Process 1978 KV load full_docs with 0 records
INFO: [] Process 1978 KV load text_chunks with 0 records
INFO: [] Process 1978 KV load full_entities with 0 records
INFO: [] Process 1978 KV load full_relations with 0 records
INFO: [] Process 1978 KV load entity_chunks with 0 records
INFO: [] Process 1978 KV load relation_chunks with 0 records
INFO: [] Process 1978 KV load llm_response_cache with 0 records
INFO: [] Process 1978 doc status load doc_status with 0 records
INFO: Processing 16 document(s)
INFO: Extracting stage 1/16: unknown_source
INFO: Processing d-id: doc-1123a80a811093ec5695cec4520ee7d2
INFO: Extracting stage 2/16: unknown_source
INFO: Processing d-id: doc-af39fe48c02d1a00ee3ef1153a44652c
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)


⚙️  Syncing LightRAG storage...
🚀 256 chunks remaining | 16 batches

📦 Batch 1/16 (size: 16)


INFO:  == LLM cache == saving: default:extract:8388aff273814093429721623d18e38f
INFO:  == LLM cache == saving: default:extract:7f7d7872f799ef097158eea9662ac915
INFO: Chunk 1 of 1 extracted 14 Ent + 13 Rel chunk-1123a80a811093ec5695cec4520ee7d2
INFO: Merging stage 1/16: unknown_source
INFO: Phase 1: Processing 14 entities from doc-1123a80a811093ec5695cec4520ee7d2 (async: 8)
INFO: Phase 2: Processing 13 relations from doc-1123a80a811093ec5695cec4520ee7d2 (async: 8)
INFO: Phase 3: Updating final 14(14+0) entities and  13 relations from doc-1123a80a811093ec5695cec4520ee7d2
INFO: Completed merging: 14 entities, 0 extra entities, 13 relations
INFO: [] Writing graph with 14 nodes, 13 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/16: unknown_source
INFO: Extracting stage 3/16: unknown_source
INFO: Processing d-id: doc-985db81e6c3d846511cb280c2983fbf0
INFO:  == LLM cache == saving: default:extract:b328c1aab5def7b1f288de9fb02bc112
ERROR: LLM func: Error in decorated 

   ✅ Done. Total: 16/256


INFO: Reset 2 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 18 document(s)
INFO: Extracting stage 1/18: unknown_source
INFO: Processing d-id: doc-af39fe48c02d1a00ee3ef1153a44652c
INFO: Extracting stage 2/18: unknown_source
INFO: Processing d-id: doc-fcb030001bc90ca154977e26797cef85
ERROR: LLM func: Error in decorated function for task 138302617140288_1238.53655034: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8574 tokens (6526 in the messages, 2048 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400}
ERROR: Failed to extract entities and relationships: C[1/1]: chunk-af39fe48c02d1a00ee3ef1153a44652c: BadRequestError: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8574 tokens (6526 in the messages, 2048 in the completion). Please re

...

   ✅ Done. Total: 240/256


INFO: Reset 34 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 50 document(s)
INFO: Extracting stage 1/50: unknown_source
INFO: Processing d-id: doc-af39fe48c02d1a00ee3ef1153a44652c
INFO: Extracting stage 2/50: unknown_source
INFO: Processing d-id: doc-fcb030001bc90ca154977e26797cef85
ERROR: LLM func: Error in decorated function for task 138299779796224_5922.025970769: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8574 tokens (6526 in the messages, 2048 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400}
ERROR: Failed to extract entities and relationships: C[1/1]: chunk-af39fe48c02d1a00ee3ef1153a44652c: BadRequestError: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8574 tokens (6526 in the messages, 2048 in the completion). Please 

   🖥️  VRAM [batch 15/16]: [░░░░░░░░░░░░░░░░░░░░] 0.8% | 0.7/85.1 GB VRAM

📦 Batch 16/16 (size: 16)


ERROR: LLM func: Error in decorated function for task 138299779796800_5922.154012518: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8250 tokens (6202 in the messages, 2048 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400}
ERROR: Failed to extract entities and relationships: C[1/1]: chunk-90e185bc8fe27d0fdacc52c2545d22fa: BadRequestError: Error code: 400 - {'object': 'error', 'message': "This model's maximum context length is 8192 tokens. However, you requested 8250 tokens (6202 in the messages, 2048 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400} (Original exception could not be reconstructed: APIStatusError.__init__() missing 2 required keyword-only arguments: 'response' and 'body')
ERROR: Traceback (most recent call last):
  File "/usr/local

   ✅ Done. Total: 256/256
   🖥️  VRAM [batch 16/16]: [░░░░░░░░░░░░░░░░░░░░] 0.8% | 0.7/85.1 GB VRAM

🏁 Indexing finished!
   ✅ All chunks indexed successfully!


## CELL 11 — Retry Failed Chunks (chỉ chạy nếu Cell 10 có lỗi)

In [ ]:
# ============================================================
# CELL 11: Retry Failed Chunks
# ============================================================
import json

if not CHECKPOINT_FILE.exists():
    print("ℹ No checkpoint — run Cell 10 first.")
else:
    with open(CHECKPOINT_FILE) as f:
        chk = json.load(f)

    failed_texts     = chk.get("failed_texts", [])
    completed_hashes = set(chk.get("completed_hashes", []))

    if not failed_texts:
        print(" No failed chunks!")
    else:
        print(f" Retrying {len(failed_texts)} chunks...")
        recovered, still_failed = [], []
        for i, text in enumerate(failed_texts):
            try:
                await rag.ainsert([text])
                completed_hashes.add(doc_hash(text))
                recovered.append(text)
                print(f"    {i+1}/{len(failed_texts)} recovered")
            except Exception as e:
                still_failed.append(text)
                print(f"    {i+1}/{len(failed_texts)} still failing: {e}")

        save_checkpoint(completed_hashes, still_failed)
        print(f"\n Recovered: {len(recovered)} | Still failed: {len(still_failed)}")
        gpu_tracker.log("after retry")


## CELL 12 — Validate Graph & Test Queries

In [26]:
# ============================================================
# CELL 12: Validate Knowledge Graph + Smoke Queries
# ============================================================
from pathlib import Path

print(" Output files:")
for f in sorted(Path(str(WORKING_DIR)).iterdir()):
    print(f"   {f.name:45s} {f.stat().st_size/1e6:8.2f} MB")

graphml = Path(str(WORKING_DIR)) / "graph_chunk_entity_relation.graphml"
if graphml.exists():
    import networkx as nx
    G = nx.read_graphml(str(graphml))
    print(f"\n Knowledge Graph:")
    print(f"   Nodes : {G.number_of_nodes()}")
    print(f"   Edges : {G.number_of_edges()}")
    ec = {}
    for _, d in G.nodes(data=True):
        t = d.get("entity_type", "?")
        ec[t] = ec.get(t, 0) + 1
    for t, c in sorted(ec.items(), key=lambda x: -x[1]):
        print(f"      {t:35s}: {c}")
else:
    print("  graph_chunk_entity_relation.graphml not found yet.")

print(f"\n{gpu_tracker.status()}")
print("\n" + "="*70)
print(" Test queries")
test_queries = [
    "Quyền và nghĩa vụ của bên mua trong hợp đồng mua bán tài sản?",
    "Chế tài khi doanh nghiệp không đăng ký kinh doanh đúng hạn?",
    "Điều kiện để hợp đồng có hiệu lực theo Bộ luật Dân sự 2015?",
]
for i, q in enumerate(test_queries, 1):
    print(f"\n {i}. {q}")
    try:
        ans = await rag.aquery(q, param=QueryParam(mode="hybrid"))
        print(f" {ans[:400]}{'...' if len(ans)>400 else ''}")
        gpu_tracker.log(f"query {i}")
    except Exception as e:
        print(f"  {e}")


INFO: Query nodes: bên mua, tài sản (top_k:40, cosine:0.2)


 Output files:
   graph_chunk_entity_relation.graphml               2.44 MB
   kv_store_doc_status.json                          0.03 MB
   kv_store_entity_chunks.json                       0.29 MB
   kv_store_full_docs.json                           1.20 MB
   kv_store_full_entities.json                       1.04 MB
   kv_store_full_relations.json                      1.30 MB
   kv_store_relation_chunks.json                     0.33 MB
   kv_store_text_chunks.json                         1.26 MB
   vdb_chunks.json                                   1.99 MB
   vdb_entities.json                                 7.60 MB
   vdb_relationships.json                            9.80 MB

 Knowledge Graph:
   Nodes : 2138
   Edges : 2754
      QUYỀN_NGHĨA_VỤ                     : 904
      CHỦ_THỂ                            : 541
      HÀNH_VI                            : 377
      UNKNOWN                            : 168
      VĂN_BẢN_PHÁP_LUẬT                  : 148

[░░░░░░░░░░░░░░░░░░░░] 1.2%

INFO: Local query: 40 entites, 166 relations
INFO: Query edges: hợp đồng mua bán, quyền và nghĩa vụ (top_k:40, cosine:0.2)
INFO: Global query: 53 entites, 40 relations
INFO: Raw search results: 85 entities, 195 relations, 0 vector chunks
INFO: After truncation: 58 entities, 138 relations
INFO: Selecting 115 from 115 entity-related chunks by vector similarity
INFO: Find 11 additional chunks in 9 relations (deduplicated 43)
INFO: Selecting 11 from 11 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 126 -> 126 (deduplicated 0)
INFO: Final context: 58 entities, 138 relations, 9 chunks
INFO: Final chunks S+F/O: E7/1 R1/1 E4/2 R1/2 E3/3 R1/3 E4/4 R1/4 E3/5
INFO:  == LLM cache == saving: hybrid:query:28dfbecfc27fb2901783a4e1170e0876
INFO: Query nodes: doanh nghiệp, không đăng ký, kỳ hạn (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 132 relations
INFO: Query edges: Chế tài, Đăng ký kinh doanh (top_k:40, cosine:0.2)
INFO: Global query: 38 entites, 40 relatio

 The rights and obligations of the buyer (Bên Mua) in a sales contract (Hợp đồng mua bán tài sản) are outlined in several legal provisions. According to Điều 444 and Điều 445, the buyer has the following rights and obligations:

### Rights of the Buyer
1. **Protection of Ownership Rights**: The buyer has the right to ensure that the purchased property is not contested by a third party. If a third p...
  VRAM [query 1]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM

 2. Chế tài khi doanh nghiệp không đăng ký kinh doanh đúng hạn?


INFO: Final context: 59 entities, 125 relations, 9 chunks
INFO: Final chunks S+F/O: E14/1 E4/2 E5/3 E6/4 E2/5 E3/6 E3/7 E4/8 E2/9
INFO:  == LLM cache == saving: hybrid:query:a7adc969ce9061bd1f12d5efc67faf20
INFO: Query nodes: điều kiện, 2015 (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 73 relations
INFO: Query edges: hợp đồng, hiệu lực, Bộ luật Dân sự (top_k:40, cosine:0.2)
INFO: Global query: 52 entites, 40 relations
INFO: Raw search results: 91 entities, 112 relations, 0 vector chunks
INFO: After truncation: 66 entities, 112 relations
INFO: Selecting 103 from 103 entity-related chunks by vector similarity
INFO: Find 7 additional chunks in 7 relations (deduplicated 42)
INFO: Selecting 7 from 7 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 110 -> 110 (deduplicated 0)
INFO: Final context: 66 entities, 112 relations, 9 chunks
INFO: Final chunks S+F/O: E3/1 R1/1 E2/2 R1/2 E5/3 R1/3 E3/4 R1/4 E3/5


 Khi doanh nghiệp không đăng ký kinh doanh đúng hạn, sẽ có một số chế tài áp dụng. Cụ thể, doanh nghiệp phải thông báo bằng văn bản cho Cơ quan đăng ký kinh doanh chậm nhất là 03 ngày làm việc trước ngày tạm ngừng kinh doanh hoặc tiếp tục kinh doanh trước thời hạn đã thông báo ([1]). Nếu doanh nghiệp không tuân thủ quy định này, Cơ quan đăng ký kinh doanh có quyền yêu cầu doanh nghiệp tạm ngừng, đì...
  VRAM [query 2]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM

 3. Điều kiện để hợp đồng có hiệu lực theo Bộ luật Dân sự 2015?


INFO:  == LLM cache == saving: hybrid:query:5d824439f74d36d4f5e799c77a755794


 Theo Bộ luật Dân sự 2015, để một hợp đồng có hiệu lực, cần tuân thủ một số điều kiện cụ thể:

1. **Hình thức hợp đồng**: Hợp đồng phải tuân thủ các quy định về hình thức. Nếu hợp đồng không tuân thủ quy định về hình thức, nó có thể bị coi là vô hiệu, trừ trường hợp một bên hoặc các bên đã thực hiện ít nhất hai phần ba nghĩa vụ trong giao dịch (Điều 129).

2. **Nội dung hợp đồng**: Hợp đồng phải tu...
  VRAM [query 3]: [░░░░░░░░░░░░░░░░░░░░] 1.2% | 1.0/85.1 GB VRAM


In [17]:
from pathlib import Path
import networkx as nx

graphml = WORKING_DIR / "graph_chunk_entity_relation.graphml"
G = nx.read_graphml(str(graphml))
print(f"📊 Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

ec = {}
for _, d in G.nodes(data=True):
    t = d.get("entity_type", "?")
    ec[t] = ec.get(t, 0) + 1
for t, c in sorted(ec.items(), key=lambda x: -x[1]):
    print(f"   {t}: {c}")

📊 Nodes: 2138 | Edges: 2754
   QUYỀN_NGHĨA_VỤ: 904
   CHỦ_THỂ: 541
   HÀNH_VI: 377
   UNKNOWN: 168
   VĂN_BẢN_PHÁP_LUẬT: 148


## CELL 13 — Backup to Google Drive

In [27]:
# ============================================================
# CELL 13: Reconstruct missing KV files + Backup toàn bộ lên Drive
#
# Bước 1 — Reconstruct: Parse graph_chunk_entity_relation.graphml
#   - kv_store_full_entities.json   (node attributes từ graphml)
#   - kv_store_full_relations.json  (edge attributes từ graphml)
#   - kv_store_entity_chunks.json   (source_id mỗi node → list chunk_ids)
#   - kv_store_relation_chunks.json (source_id mỗi edge → list chunk_ids)
#
# Bước 2 — Backup: Copy TẤT CẢ file trong WORKING_DIR lên Drive
#   (toàn bộ thư mục, không hardcode danh sách → không bao giờ miss file)
# ============================================================
import json, hashlib, shutil
import networkx as nx
from datetime import datetime
from pathlib import Path

# ── BƯỚC 1: Reconstruct missing KV files từ graphml ──────────────────────────
print(" Reconstructing missing KV files from graph...\n")

graphml_path = WORKING_DIR / "graph_chunk_entity_relation.graphml"
assert graphml_path.exists(), f" graphml not found: {graphml_path}"

G = nx.read_graphml(str(graphml_path))
print(f"   Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# 1a. kv_store_full_entities — node_id → {entity_type, description, source_id, ...}
kv_full_entities = {}
for node_id, data in G.nodes(data=True):
    kv_full_entities[node_id] = {
        "entity_id":   data.get("entity_id",   node_id),
        "entity_type": data.get("entity_type", "UNKNOWN"),
        "description": data.get("description", ""),
        "source_id":   data.get("source_id",   ""),
        "file_path":   data.get("file_path",   "unknown_source"),
        "created_at":  data.get("created_at",  0),
        "truncate":    data.get("truncate",    ""),
    }

# 1b. kv_store_full_relations — rel_id → {src_id, tgt_id, description, ...}
kv_full_relations = {}
for src, tgt, data in G.edges(data=True):
    rel_hash = hashlib.md5(f"{src.lower()}{tgt.lower()}".encode()).hexdigest()
    rel_id   = f"rel-{rel_hash}"
    kv_full_relations[rel_id] = {
        "src_id":      src,
        "tgt_id":      tgt,
        "description": data.get("description", ""),
        "keywords":    data.get("keywords",    ""),
        "weight":      data.get("weight",      1.0),
        "source_id":   data.get("source_id",   ""),
        "file_path":   data.get("file_path",   "unknown_source"),
        "created_at":  data.get("created_at",  0),
        "truncate":    data.get("truncate",    ""),
    }

# 1c. kv_store_entity_chunks — node_id → {chunks: [chunk_id, ...]}
kv_entity_chunks = {}
for node_id, data in G.nodes(data=True):
    source_id = data.get("source_id", "")
    chunk_ids = [s.strip() for s in source_id.split("<SEP>") if s.strip()]
    kv_entity_chunks[node_id] = {"chunks": chunk_ids}

# 1d. kv_store_relation_chunks — rel_id → {chunks: [chunk_id, ...]}
kv_relation_chunks = {}
for src, tgt, data in G.edges(data=True):
    rel_hash  = hashlib.md5(f"{src.lower()}{tgt.lower()}".encode()).hexdigest()
    rel_id    = f"rel-{rel_hash}"
    source_id = data.get("source_id", "")
    chunk_ids = [s.strip() for s in source_id.split("<SEP>") if s.strip()]
    kv_relation_chunks[rel_id] = {"chunks": chunk_ids}

# Ghi các file đã reconstruct vào WORKING_DIR
reconstructed = {
    "kv_store_full_entities.json":    kv_full_entities,
    "kv_store_full_relations.json":   kv_full_relations,
    "kv_store_entity_chunks.json":    kv_entity_chunks,
    "kv_store_relation_chunks.json":  kv_relation_chunks,
}

for fname, data in reconstructed.items():
    fpath = WORKING_DIR / fname
    with open(fpath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"    {fname}: {len(data)} records ({fpath.stat().st_size/1e6:.2f} MB)")

print(f"\n   Reconstructed: {len(reconstructed)} files")

# ── BƯỚC 2: Backup toàn bộ WORKING_DIR lên Drive ─────────────────────────────
print("\n Backing up ALL files to Google Drive...")

ts         = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = GDRIVE_BASE / f"backups/run_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

print(f"   Source : {WORKING_DIR}")
print(f"   Dest   : {backup_dir}")

copied = 0
for src in sorted(WORKING_DIR.iterdir()):
    if not src.is_file():
        continue
    shutil.copy2(src, backup_dir / src.name)
    print(f"    {src.name:45s} {src.stat().st_size/1e6:6.2f} MB")
    copied += 1

# ── Tổng kết ──────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print(" PHASE 2 COMPLETE")
print(f"   Reconstructed : {len(reconstructed)} missing KV files")
print(f"   Backed up     : {copied} files total")
print(f"   Backup dir    : {backup_dir}")
print(f"   {gpu_tracker.status()}")
print("="*70)

 Reconstructing missing KV files from graph...

   Graph loaded: 2138 nodes, 2754 edges
    kv_store_full_entities.json: 2138 records (1.04 MB)
    kv_store_full_relations.json: 2686 records (1.30 MB)
    kv_store_entity_chunks.json: 2138 records (0.29 MB)
    kv_store_relation_chunks.json: 2686 records (0.33 MB)

   Reconstructed: 4 files

 Backing up ALL files to Google Drive...
   Source : /content/drive/MyDrive/viet_auditor/lightrag_index
   Dest   : /content/drive/MyDrive/viet_auditor/backups/run_20260319_025236
    graph_chunk_entity_relation.graphml             2.44 MB
    kv_store_doc_status.json                        0.03 MB
    kv_store_entity_chunks.json                     0.29 MB
    kv_store_full_docs.json                         1.20 MB
    kv_store_full_entities.json                     1.04 MB
    kv_store_full_relations.json                    1.30 MB
    kv_store_llm_response_cache.json                0.01 MB
    kv_store_relation_chunks.json                   0.33 

## CELL 14 — ⚠️ Terminate A100 Runtime (Run LAST)

In [28]:
# ============================================================
# CELL 14: Release A100 Compute Units
# ⚠️  Chỉ chạy SAU KHI Cell 13 backup xong!
# ============================================================
import time
print("⚠️  Releasing A100 in 5s — interrupt NOW to cancel!")
for i in range(5, 0, -1):
    print(f"   {i}...", end=" ", flush=True)
    time.sleep(1)
print("\n🔌 Shutting down...")
from google.colab import runtime
runtime.unassign()


⚠️  Releasing A100 in 5s — interrupt NOW to cancel!
   5...    4...    3...    2...    1... 
🔌 Shutting down...
